In [2]:
#!/usr/bin/env python3
"""
ingest_momentum_secuencial_auditado.py

Cálculo SECUENCIAL de Momentum 12m-1m (FMP historical prices)
+ auditoría completa en infraestructura.update_logs
"""

import os
import time
import logging
import requests
import subprocess
import pandas as pd
from datetime import datetime, date
from dotenv import load_dotenv
import psycopg2

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
API_KEY = os.getenv("FMP_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

BASE_URL = "https://financialmodelingprep.com/stable/historical-price-eod/dividend-adjusted"

# ---------------------------
# Logging
# ---------------------------
LOG_DIR = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/ingest_momentum_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("=== START ingest_momentum_secuencial ===")

# ---------------------------
# Keep Mac awake
# ---------------------------
try:
    caffeinate = subprocess.Popen(["caffeinate"])
except Exception:
    caffeinate = None

# ---------------------------
# DB helpers
# ---------------------------
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        host=POSTGRES_HOST,
        port=POSTGRES_PORT
    )

def log_db(conn, ticker, status, message):
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO infraestructura.update_logs
            (schema_name, table_name, ticker, status, message)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            'api_raw',
            'multifactor_momentum',
            ticker,
            status,
            message
        ))
        conn.commit()

# ---------------------------
# Fetch tickers
# ---------------------------
def obtener_tickers(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT ticker FROM universos.stock_opciones_2026")
        return [r[0] for r in cur.fetchall()]

# ---------------------------
# Momentum logic
# ---------------------------
def fetch_momentum_12m_1m(ticker):
    url = f"{BASE_URL}?symbol={ticker}&apikey={API_KEY}"
    r = requests.get(url, timeout=30)

    if r.status_code != 200:
        return None, f"HTTP_{r.status_code}"

    data = r.json()
    if not data or len(data) < 260:
        return None, "INSUFFICIENT_HISTORY"

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values("date", inplace=True)

    df_monthly = df.resample("M", on="date").last()
    if len(df_monthly) < 13:
        return None, "NO_DATA"

    price_1m = float(df_monthly["adjClose"].iloc[-2])
    price_12m = float(df_monthly["adjClose"].iloc[-13])

    if price_12m <= 0:
        return None, "INVALID_PRICE"

    momentum = (price_1m / price_12m) - 1

    return {
        "ticker": ticker,
        "fecha": date.today(),
        "price_1m": price_1m,
        "price_12m": price_12m,
        "momentum": momentum,
        "source": "FMP_historical_price_eod_dividend_adjusted"
    }, "OK"

# ---------------------------
# SQL
# ---------------------------
INSERT_SQL = """
INSERT INTO api_raw.multifactor_momentum (
    ticker, fecha_de_consulta,
    price_1m, price_12m,
    momentum_12m_1m,
    source
)
VALUES (
    %(ticker)s, %(fecha)s,
    %(price_1m)s, %(price_12m)s,
    %(momentum)s,
    %(source)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    price_1m = EXCLUDED.price_1m,
    price_12m = EXCLUDED.price_12m,
    momentum_12m_1m = EXCLUDED.momentum_12m_1m,
    source = EXCLUDED.source,
    updated_at = CURRENT_TIMESTAMP;
"""

# ---------------------------
# MAIN
# ---------------------------
def main():
    conn = get_conn()
    tickers = obtener_tickers(conn)
    total = len(tickers)

    logging.info(f"Tickers a procesar: {total}")

    ok, fail = 0, 0

    for i, ticker in enumerate(tickers, 1):
        try:
            data, status = fetch_momentum_12m_1m(ticker)

            if not data:
                log_db(conn, ticker, status, "Momentum fetch failed")
                fail += 1
                continue

            with conn.cursor() as cur:
                cur.execute(INSERT_SQL, data)
                conn.commit()

            log_db(conn, ticker, "success", "Inserted momentum")
            ok += 1

        except Exception as e:
            conn.rollback()
            log_db(conn, ticker, "exception", str(e))
            fail += 1
            logging.warning(f"{ticker} error: {e}")

        if i % 100 == 0:
            logging.info(f"Procesados {i}/{total}")

        time.sleep(0.30)

    conn.close()
    logging.info(f"FIN | OK={ok} | FAIL={fail}")

# ---------------------------
if __name__ == "__main__":
    try:
        main()
    finally:
        if caffeinate:
            caffeinate.terminate()


2026-04-06 19:50:05,352 | INFO | === START ingest_momentum_secuencial ===
2026-04-06 19:50:05,478 | INFO | Tickers a procesar: 3027
2026-04-06 19:52:06,873 | INFO | Procesados 100/3027
2026-04-06 19:54:07,994 | INFO | Procesados 200/3027
2026-04-06 19:56:09,813 | INFO | Procesados 300/3027
2026-04-06 19:58:12,357 | INFO | Procesados 400/3027
2026-04-06 20:00:21,217 | INFO | Procesados 500/3027
2026-04-06 20:02:22,634 | INFO | Procesados 600/3027
2026-04-06 20:04:26,027 | INFO | Procesados 700/3027
2026-04-06 20:06:29,665 | INFO | Procesados 800/3027
2026-04-06 20:08:31,153 | INFO | Procesados 900/3027
2026-04-06 20:10:33,751 | INFO | Procesados 1000/3027
2026-04-06 20:12:35,248 | INFO | Procesados 1100/3027
2026-04-06 20:14:36,402 | INFO | Procesados 1200/3027
2026-04-06 20:16:39,566 | INFO | Procesados 1300/3027
2026-04-06 20:18:42,365 | INFO | Procesados 1400/3027
2026-04-06 20:20:44,148 | INFO | Procesados 1500/3027
2026-04-06 20:22:45,040 | INFO | Procesados 1600/3027
2026-04-06 20